In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import math
import argparse
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from collections import OrderedDict
from matplotlib.collections import LineCollection

import torch

from import_model import import_model
from import_dataset import import_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
train_session_dir = "train_sessions/EncoderDecoderGRU_20250811_211906"

In [ ]:
json_to_inference = os.path.join(train_session_dir,"args.json")
with open(json_to_inference, 'r') as f:
    data = json.load(f)

model_name = data["model_name"]
train_pred_name = f'{model_name}_train_pred_series.pt'
val_pred_name = f'{model_name}_val_pred_series.pt'

target_and_pred_names = [train_pred_name, val_pred_name]
model_pt_name = [x for x in os.listdir(train_session_dir) if x.endswith(".pt") and x not in target_and_pred_names][0]
model_pt_path = os.path.join(train_session_dir, model_pt_name)

args = argparse.Namespace(**data)

input_cols = [f'{c}_{f}' for c in args.input_coins for f in args.input_features]
output_cols = [f'{c}_{f}' for c in args.output_coins for f in args.output_features]
output_col_indices_in_input_cols = [input_cols.index(col) for col in output_cols]
target_coin_indices = [args.input_coins.index(c) for c in args.output_coins]

model_kwargs = {"input_features": len(args.input_coins)*len(args.input_features), "output_features": len(args.output_coins)*len(args.output_features),
"input_window": args.input_window, "output_window": args.output_window,
"dropout": args.dropout, "num_layers": args.num_layers, "hidden_dim": args.hidden_dim, "num_heads": args.num_heads,
"teacher_forcing_ratio": args.teacher_forcing_ratio, "input_cols":input_cols, "output_cols":output_cols,
"target_coin_indices": target_coin_indices, "output_col_indices_in_input_cols": output_col_indices_in_input_cols,
"device": device}

model = import_model(args.model_name, **model_kwargs)
model.load_state_dict(torch.load(model_pt_path, weights_only=True))
if hasattr(model, "set_teacher_forcing_ratio"):
    model.set_teacher_forcing_ratio(0)

base_dataset_kwargs = {"input_coins": args.input_coins, "input_features": args.input_features, "output_coins": args.output_coins,
"output_features": args.output_features, "input_window": args.input_window, "output_window": args.output_window,
"augmentation_noise_std": args.augmentation_noise_std, "augmentation_constant_c": args.augmentation_constant_c, "augmentation_scale_s": args.augmentation_scale_s,
"transform_name":args.transform_name, "output_distribution": args.output_distribution, "n_quantiles": args.n_quantiles, "train_session_dir": train_session_dir}

val_inference_dataset_kwargs = {**base_dataset_kwargs, "csv_path": args.val_csv_path, "augmentation_p": 0, "training_dataset":0}
val_inference_dataset = import_dataset(args.dataset_name, **val_inference_dataset_kwargs)

train_inference_dataset_kwargs = {**base_dataset_kwargs, "csv_path": args.train_csv_path, "augmentation_p": 0, "training_dataset":1}
train_inference_dataset = import_dataset(args.dataset_name, **train_inference_dataset_kwargs)

# MODEL'S PREDICTIONS
train_pred_path = os.path.join(train_session_dir, train_pred_name)
val_pred_path = os.path.join(train_session_dir, val_pred_name)

train_pred_series = torch.load(train_pred_path)
val_pred_series = torch.load(val_pred_path)

# DATA THE MODEL SAW DURING THE TRAINING
train_learned_dataframe_crop = train_inference_dataset.df.iloc[data["input_window"]:]
val_learned_dataframe_crop = val_inference_dataset.df.iloc[data["input_window"]:]

# REAL PRICE DATA
train_df = pd.read_csv(data["train_csv_path"], index_col="open_time")
train_df_preds = train_df.loc[train_learned_dataframe_crop.index]
# we take the data["input_window"]th data as the initial price instead of data["input_window"]-1
# it's because we are shifting the dates 1 interval back during the preprocessing, so the first day is kinda wasted for the lof return computations
# and so the initial price is also shifted one interval further than it would be at an unshifted dataset
t1 = datetime.strptime(train_learned_dataframe_crop.index[0], "%Y-%m-%d %H:%M:%S")
t2 = datetime.strptime(train_learned_dataframe_crop.index[1], "%Y-%m-%d %H:%M:%S")
delta = t2 - t1
t0 = t1 - delta
train_initial_prices = torch.tensor(train_df.loc[str(t0)][output_cols].values).unsqueeze(0)

val_df = pd.read_csv(data["val_csv_path"], index_col="open_time")
val_df_preds = val_df.loc[val_learned_dataframe_crop.index]
t1 = datetime.strptime(val_learned_dataframe_crop.index[0], "%Y-%m-%d %H:%M:%S")
t2 = datetime.strptime(val_learned_dataframe_crop.index[1], "%Y-%m-%d %H:%M:%S")
delta = t2 - t1
t0 = t1 - delta
val_initial_prices = torch.tensor(val_df.loc[str(t0)][output_cols].values).unsqueeze(0)